# Background

Author: Diane Menuz  
Date: 2026-01-26  
Station: Cedar Mesa

Combine preprocessed data together and fix any alignment issues

# Set Up

## Parameters

In [ ]:
station = "US-UTJ"
interval = 30 # data interval, useful when pulling in data based on naming conventions

amflux_file = r'M:\Shared drives\UGS_Flux\Data_Processing\site_specific_data_review\flux-met_processing_variables_20250818.csv' # file with names of amflux variables
parquet_path = r'M:\Shared drives\UGS_Flux\Data_Downloads\compiled\preprocessed_site_data' #path for parquet files of compiled data to import
clean_file_folder = r'M:\Shared drives\UGS_Flux\Data_Processing\final_database_tables\\' #path where cleaned data will be output
micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"

## Libraries

In [ ]:
import sys

sys.path.append(micromet_path)
from micromet import validate
from micromet import validation
from micromet import merge
from micromet import data_cleaning
from micromet import columns
from micromet import eddy_plots as ed_plot

In [ ]:
import pandas as pd
import numpy as np
import importlib

from prettytable import PrettyTable
from typing import Union

from scipy import stats
import matplotlib.pyplot as plt


from typing import List, Dict, Union

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from plotly.offline import iplot

## Database Info

In [ ]:
import requests
headdict = {'Accept-Profile': 'groundwater',  
            'Content-Type': 'application/json'}  

table = 'eddy_events'
resp = requests.get(f"https://ugs-koop-umfdxaxiyq-wm.a.run.app/{table}",  
                    headers=headdict)
event_db = pd.DataFrame(resp.json())
event_db['event_date'] = pd.to_datetime(event_db['event_date'])

table = 'eddy_station_metadata'
resp = requests.get(f"https://ugs-koop-umfdxaxiyq-wm.a.run.app/{table}",  
                    headers=headdict)
metadata = pd.DataFrame(resp.json())

# Site specific vars and review of notes

In [ ]:
# set up the stationid and review the install date
install_date = metadata.loc[metadata.stationid==station, 'install_date']
install_date = pd.to_datetime(install_date).iloc[0]
print(f'{station}, installed on {install_date}')

In [ ]:
notes = event_db.loc[(event_db.stationid==station) & (event_db.event_type=='station visit'),
                     ['event_date','event_notes']]

table = PrettyTable()
table.field_names = ["Date", "Note"]
table.sortby = "Date"
table.align["Note"] = "l"

rows_as_tuples = [tuple(x) for x in notes.values] 
table.add_rows(rows_as_tuples) # Use the correct data format

print(table)

In [ ]:
# review program updates
updates = event_db.loc[(event_db.stationid==station) & 
                       (event_db.event_type=='program update'),
                       ['event_date','event_notes']]

table = PrettyTable()
table.field_names = ["Date", "Note"]
table.sortby = "Date"
table.align["Note"] = "l"

rows_as_tuples = [tuple(x) for x in updates.values] 
table.add_rows(rows_as_tuples) # Use the correct data format

print(table)

# Eddy wrangling

Need to determine which files to read in, based on obtaining a complete dataset, including values  
only found in csflux data (like the G_PLATE values). You can just read in data from the web downloads  
if those files are complete.

## CSFlux

In [ ]:
temp_csflux_web = pd.read_parquet(f"{parquet_path}/{station}_{interval}_eddycsflux_web_preprocessed.parquet")
temp_csflux_dl = pd.read_parquet(f"{parquet_path}/{station}_{interval}_eddycsflux_dl_preprocessed.parquet")
csfluxdl = data_cleaning.prep_parquet(station, temp_csflux_dl)
csfluxweb = data_cleaning.prep_parquet(station, temp_csflux_web)

In [ ]:
# look at web vs dl- hopefully full overlap so we aren't missing any variables!!
round_digit = 5

print(f'Percent of rows with differences between web and datalogger data rounded to {round_digit} digits')
x = validate.data_diff_check(csfluxdl.round(round_digit), csfluxweb.round(round_digit))
print(x[x.percent_different>0.5])

ed_plot.plotlystuff([csfluxdl, csfluxweb], ['WD_1_1_1', 'WD_1_1_1'])

In [ ]:
csflux = merge.fillna_with_second_df(csfluxweb, csfluxdl)
ed_plot.plotlystuff([csflux], ['WD_1_1_1'])

## Ameriflux

In [ ]:
temp_af_web = pd.read_parquet(f"{parquet_path}/{station}_{interval}_eddyaf_web_preprocessed.parquet")
temp_af_dl = pd.read_parquet(f"{parquet_path}/{station}_{interval}_eddyaf_dl_preprocessed.parquet")
afweb = data_cleaning.prep_parquet(station, temp_af_web)
afweb = afweb.rename(columns = {'T_SONIC':'T_SONIC_1_1_1'}) # renamining this one so it will match with the csflux data
afdl = data_cleaning.prep_parquet(station, temp_af_dl)
afdl = afdl.rename(columns = {'T_SONIC':'T_SONIC_1_1_1'}) # renamining this one so it will match with the csflux data

In [ ]:
# look at web vs dl- hopefully full overlap so we aren't missing any variables!!
round_digit =3

print(f'Percent of rows with differences between web and datalogger data rounded to {round_digit} digits')
x = validate.data_diff_check(afweb.round(round_digit), afdl.round(round_digit))
print(x[x.percent_different>0.5].round(1))

ed_plot.plotlystuff([afweb, afdl], ['WS_1_1_1', 'WS_1_1_1'])

In [ ]:
# you can do the following to review a variable that difference from another variable
round_val = 2
col = 'CO2_1_1_1'
df1 = afweb.copy()
df2 = afdl.copy()
df1 = df1.dropna(subset = [col]).round(round_val)
df2 = df2.dropna(subset = [col]).round(round_val)
df1_prep, df2_prep = validate.prep_for_comparison(df1, df2)
x = df1_prep[col].compare(df2_prep[col])
print(x)

In [ ]:
amflux = merge.fillna_with_second_df(afweb, afdl)
ed_plot.plotlystuff([amflux], ['WD_1_1_1'])

## Final Eddy Combination

In [ ]:
round_digit = 5

print(f'Percent of rows with differences between amflux and csflux data rounded to {round_digit} digits')
x = validate.data_diff_check(amflux.round(round_digit), csflux.round(round_digit))
print(x[x.percent_different>0.5].round(1))

ed_plot.plotlystuff([amflux, csflux], ['WS_1_1_1', 'WS_1_1_1'])

In [ ]:
# combine ameriflux and csflux data

# csflux has a bunch of additional columns; I'll only join the columns below with the temped file
# if we do want to join all columns, will need to look at renaming some columns
csflux_join_cols = ['LE_1_1_1', 'H_1_1_1',
       'NETRAD_1_1_1', 'G_1_1_A','G_2_1_1', 'G_1_1_1',
       'SG_2_1_1', 'SG_1_1_1','G_PLATE_2_1_1', 'G_PLATE_1_1_1',
       'SG_1_1_A', 'TAU', 'USTAR', 'TA_1_1_1',
       'RH_1_1_1', 'T_DP_1_1_1', 'TA_1_2_1', 'RH_1_2_1', 'T_DP_1_2_1',
       'TA_1_3_1', 'RH_1_3_1', 'T_DP_1_3_1', 'PA_1_1_1', 'VPD_1_1_1',
       'TA_1_4_1', 'T_SONIC_SIGMA', 'WS_1_1_1', 'WD_1_1_1', 'WS_MAX_1_1_1',
       'CO2_SIG_STRGTH_MIN', 'H2O_SIG_STRGTH_MIN', 'P_1_1_1', 'ALB_1_1_1',
       'SW_IN_1_1_1', 'SW_OUT_1_1_1', 'LW_IN_1_1_1', 'LW_OUT_1_1_1','T_SONIC_1_1_1',
       'TS_1_1_1', 'TS_2_1_1', 'SWC_1_1_1', 'SWC_2_1_1', 'FETCH_MAX',
       'FETCH_90', 'FETCH_55', 'FETCH_40', 'R_LW_IN_MEAS', 'R_LW_OUT_MEAS',
       'FILE_NAME',
       'CO2_DENSITY', 'CO2_DENSITY_SIGMA', 
       'FC_MASS', 'FC_QC', 'FC_SAMPLES', 'FP_DIST_INTRST',
       'H2O_DENSITY', 'H2O_DENSITY_SIGMA', 'H_QC',
       'H_SAMPLES', 'LE_QC', 'LE_SAMPLES', 'TAU_QC', 
       'TKE', 'TSTAR',
       'T_NR', 'UPWND_DIST_INTRST', 'UX', 'UX_SIGMA', 'UY', 'UY_SIGMA', 'UZ',
       'UZ_SIGMA','WD_SIGMA', 'WD_SONIC']

print('Columns not moved into final dataset from csflux')
# note that EC and TS values in this list are likely from the T65x (not the TCAV)
print(csflux.columns[~csflux.columns.isin(csflux_join_cols)].sort_values())
print('Columns in csflux that are also in easyflux not moved into final dataset')
print([col for col in csflux.columns if col in amflux.columns and col not in csflux_join_cols])
print('\n')


eddy = merge.fillna_with_second_df(amflux, csflux[csflux_join_cols])

ed_plot.plotlystuff([eddy, eddy], 
                    ['WD_1_1_1', 'G_PLATE_1_1_1'], 
                    chrttitle='Two-variable review for missing eddy data')

In [ ]:
amflux = pd.read_csv(amflux_file)
name_comparison = validate.compare_names_to_ameriflux(eddy, amflux)

# Met wrangling

In [ ]:
temp_metstat = pd.read_parquet(f"{parquet_path}/{station}_{interval}_metstats_preprocessed.parquet")
temp_metaf = pd.read_parquet(f"{parquet_path}/{station}_{interval}_metstatsaf_preprocessed.parquet")

In [ ]:
metstat = data_cleaning.prep_parquet(station, temp_metstat)
metaf = data_cleaning.prep_parquet(station, temp_metaf)

In [ ]:
# comparison using rounded data
# at cedar mesa, any issues are either soil vue or rounding diff
round = 1
x = validate.data_diff_check(metstat.round(round), metaf.round(round))
print(x[x.percent_different>.5].sort_index())

In [ ]:
ed_plot.plotlystuff([metstat, metaf], ['NETRAD_1_1_2','NETRAD_1_1_2'])
ed_plot.plotlystuff([metstat, metaf], ['WD_1_1_2','WD_1_1_2'])

## Fix Met Shift

In [ ]:
# review data lags using temp sensors
print ("Lag for the two met temp sensors")
f = validate.review_lags(metstat.SWC_3_1_1, metaf.SWC_3_1_1, 2)
print('\n')

# review for lag
swc_offset = validate.detect_sectional_offsets_indexed(metstat, metaf,
                                          'SWC_3_1_1', 'SWC_3_1_1')
validate.plot_sectional_lags_plotly(swc_offset)

In [ ]:
ed_plot.plotlystuff([metstat, metaf], ['TS_3_1_1','TS_3_1_1'])

In [ ]:
# address soilvue shift
# after shifting data, review plots to make sure shift looks correct
columns_to_shift = metaf.columns[metaf.columns.str.contains('EC_')].append(
    metaf.columns[metaf.columns.str.contains('K_')]).append(
        metaf.columns[metaf.columns.str.contains('SWC_')]).append(
            metaf.columns[metaf.columns.str.contains('TS_')])

# actually making the data shift
metaf_shifted = metaf[columns_to_shift].shift(freq = '30min')
metaf_fixed = metaf.copy()
metaf_fixed.loc[:, columns_to_shift] = metaf_shifted

# subsetting small section of data to review in plots
subdate = pd.to_datetime('2025-08-25')
subdate2 = pd.to_datetime('2025-09-05')
sub1 = metstat[(metstat.index>subdate) & (metstat.index<subdate2)]
sub2 = metaf_fixed[(metaf_fixed.index>subdate) & (metaf_fixed.index<subdate2)]
sub3 = metaf[(metaf.index>subdate) & (metaf.index<subdate2)]
ed_plot.plotlystuff([sub1, sub3], ['TS_3_1_1', 'TS_3_1_1'], 
                    chrttitle="Check that original data mis-aligned")
ed_plot.plotlystuff([sub1, sub2], ['TS_3_1_1', 'TS_3_1_1'], 
                    chrttitle="Check that SoilVue data are now aligned")
ed_plot.plotlystuff([sub1, sub2], ['SWC_3_3_1', 'SWC_3_3_1'], 
                    chrttitle="Check that SoilVue data are now aligned")
ed_plot.plotlystuff([sub1, sub2], ['NETRAD_1_1_2', 'NETRAD_1_1_2'], 
                    chrttitle="Check that you didn't mess up netrad alignment")

In [ ]:
# still not perfect...
round_digit = 1
x = validate.data_diff_check(metstat.round(round_digit), metaf_fixed.round(round_digit))
print(x[x.percent_different>1].sort_index())

In [ ]:
for col in columns_to_shift:
    print (f"Lag for {col}")
    f = validate.review_lags(metstat[col], metaf_fixed[col], 2)
    print('\n')

In [ ]:
# you can do the following to review a variable that difference from another variable
round_val = 1
col = 'TS_3_1_1'
df1 = metaf_fixed.copy()
df2 = metstat.copy()
df1 = df1.dropna(subset = [col]).round(round_val)
df2 = df2.dropna(subset = [col]).round(round_val)
df1_prep, df2_prep = validate.prep_for_comparison(df1, df2)
x = df1_prep[col].compare(df2_prep[col])
print(x.tail(20))

## Merge Met Data Together


In [ ]:
met = merge.fillna_with_second_df(metstat, metaf_fixed)
ed_plot.plotlystuff(
    [metaf, met], ['WD_1_1_2','WD_1_1_2'], 
    chrttitle='Anything in Green shows data that will not show up in final file')

In [ ]:
amflux = pd.read_csv(amflux_file)
name_comparison = validate.compare_names_to_ameriflux(met, amflux)

In [ ]:
# drop these columns so they aren't duplicated in the later merge
met = met.drop(columns=[ 'SAMPLING_INTERVAL'])

In [ ]:
for col in met.columns.sort_values():
    print(col)

# Manage Timeshift Issues

In [ ]:
final_eddy = eddy.copy()
final_met = met.copy()

In [ ]:
# subset out chunks of data to align and view in plot
bad_met = final_met[final_met.index<=pd.to_datetime('2024-05-30')]
potential_eddy = final_eddy[final_eddy.index<pd.to_datetime('2025-02-10')]
ed_plot.plotlystuff([potential_eddy, bad_met], ['NETRAD_1_1_1', 'NETRAD_1_1_2'])

In [ ]:
# use functions to get close to lag and then inspect plots to ID correct lag
freq = 'h'
lag, corr = data_cleaning.find_optimal_shift(bad_met, potential_eddy, 'WS_1_1_2', 'WS_1_1_1', freq=freq, 
                               min_lag_units=1440, max_lag_units=7200)
print(lag, corr)
met_initial_fix = data_cleaning.apply_lag_shift(bad_met, lag, freq_unit=freq)
ed_plot.plotlystuff([potential_eddy, met_initial_fix], ['WS_1_1_1', 'WS_1_1_2'])

In [ ]:
lag2 = -13*24 + lag - 1
print(f'Final lag is {lag2} {freq}')
met_final_fix = data_cleaning.apply_lag_shift(bad_met, lag2, freq_unit=freq)
ed_plot.plotlystuff([potential_eddy, met_final_fix], ['WS_1_1_1', 'WS_1_1_2'])
ed_plot.plotlystuff([potential_eddy, met_final_fix], ['NETRAD_1_1_1', 'NETRAD_1_1_2'])

In [ ]:
# now combine two met datasets together
good_met = final_met[final_met.index.floor('D')>=install_date]
cleaned_met = pd.concat([good_met, met_final_fix]).sort_index()
ed_plot.plotlystuff([final_eddy, cleaned_met], ['NETRAD_1_1_1', 'NETRAD_1_1_2'])
ed_plot.plotlystuff([final_eddy, cleaned_met], ['WS_1_1_1', 'WS_1_1_2'])

In [ ]:
# once you are sure the data are good, turn cleaned_met back into final_met!!
final_met = cleaned_met.copy()

In [ ]:
# review data lags using temp sensors
var1 = 'WS_1_1_1'
var2 = 'WS_1_1_2'

print ("Lag for the sensors")
f = validate.review_lags(final_eddy[var1], final_met[var2], 2)
print('\n')

# review for lag
offset = validate.detect_sectional_offsets_indexed(final_eddy, final_met,
                                          var1, var2)
validate.plot_sectional_lags_plotly(offset)

# Combine Datasets and Export

In [ ]:
# show any duplicate columns between the two
[col for col in final_eddy.columns if col in final_met.columns]

In [ ]:
# rename column names if we want to keep columns in both met and eddy
# will reconcile timestamps later
final_eddy_clean = final_eddy.rename(columns={
    'FILE_NAME':'FILE_NAME_EDDY',
    'T_NR': 'T_NR_1_1_1'
    })
final_met = final_met.rename(columns={
    'FILE_NAME':'FILE_NAME_MET',
    'T_NR': 'T_NR_1_1_2'
    })

In [ ]:
# review start and end dates for met and eddy data!
# met data turned hourly after the eddy data

print(f'Eddy data: {final_eddy_clean.index.min()} to {final_eddy_clean.index.max()}')
print(f'Met data: {final_met.index.min()} to {final_met.index.max()}')

In [ ]:
# review for any lags between NETRAD and WS values
netrad_offset = validate.detect_sectional_offsets_indexed(final_eddy_clean, final_met,
                                          'NETRAD_1_1_1', 'NETRAD_1_1_2')
validate.plot_sectional_lags_plotly(netrad_offset)
ws_offset = validate.detect_sectional_offsets_indexed(final_eddy_clean, final_met,
                                          'WS_1_1_1', 'WS_1_1_2')
validate.plot_sectional_lags_plotly(ws_offset)

In [ ]:
# combine datasets
alldat = pd.merge(final_eddy_clean,
              final_met,
              left_index=True,
              right_index=True,
              how='outer')

In [ ]:
# check whether all timestamps already on 30 minute interval
minute_off = alldat.index.minute % 30 != 0
if len (alldat[minute_off])>0:
    print("Not all records on 30 minute intervals")
else:
    print("All records on 30 minute intervals")

In [ ]:
# look for any columns with _x or _y at the end- these need to be combined; see code above

x_cols = alldat.columns[alldat.columns.str.contains("_x")]

if (x_cols.size)>0:
    print("Columns duplicated between datasets")
    print(x_cols)
else:
    print('No duplicates columns from merge')

In [ ]:
# Columns to drop, RECORD from above
# also droppping G_1_1_A since it is dervied and G_3_1_1 (from SoilVue) b/c will recalculate
# this will fail if one of the columns isn't present

alldat = alldat.drop(columns = [
    'RECORD_x', 'RECORD_y', 'G_1_1_A', 'G_3_1_1', 'SG_1_1_A',
     'TIMESTAMP_START_x', 'TIMESTAMP_END_x', 'TIMESTAMP_START_y', 'TIMESTAMP_END_y'])

In [ ]:
for col in alldat.columns.sort_values():
    print(col)

In [ ]:
# rename columns- first make sure column list is complete for your site
mask = (alldat.columns.str.contains('_1')) | (alldat.columns.str.contains('_2')) 
cols_no_suffix = alldat.columns[~mask]

col_list = ['CO2_SIGMA','H2O_SIGMA','FC_SSITC_TEST','LE_SSITC_TEST',
            'ET_SSITC_TEST','H_SSITC_TEST','USTAR','ZL','TAU','TAU_SSITC_TEST',
            'MO_LENGTH','U','U_SIGMA','V', 'V_SIGMA', 'W', 'W_SIGMA', 
            'T_SONIC_SIGMA','CO2_SIG_STRGTH_MIN', 
            'H2O_SIG_STRGTH_MIN','R_LW_IN_MEAS', 'R_LW_OUT_MEAS', 
            'T_NR', 'T_NR_OUT', 'T_CANOPY', 'T_SI111_BODY',
            'PPFD_IN', 'WND_DIR_SD1_WVT', 'D_SNOW','CO2_DENSITY', 'CO2_DENSITY_SIGMA', 
            'FC_MASS','FC_QC', 'FC_SAMPLES',
            'H2O_DENSITY','H2O_DENSITY_SIGMA', 'H_QC', 'H_SAMPLES',
            'LE_QC', 'LE_SAMPLES','TAU_QC', 'TKE', 'TSTAR','UX',
            'UX_SIGMA', 'UY', 'UY_SIGMA', 'UZ', 'UZ_SIGMA', 'WD_SIGMA', 'WD_SONIC'
            ]

print('Columns that will not have a suffix added')
cols_no_suffix[~cols_no_suffix.isin(col_list)].sort_values()

In [ ]:
# this creating a renaming dictionary to add suffixes, for whichever columns are present in the data
rename_dict = columns.create_suffix_map(alldat, col_list, suffix = '_1_1_1')
alldat  = alldat.rename(columns=rename_dict)

In [ ]:
# drop any data from before the install date
min_data_date = alldat.index.min()
if alldat.index.min()<install_date:
    print(f'Date of install ({install_date}) AFTER first data date ({min_data_date}) ')
    print(f'WARNING, dropping all data before {install_date}')
    alldat = alldat[alldat.index.floor('D')>install_date]
else:
    print(f'NO ACTION NEEDED: Date of install ({install_date}) before first data date ({min_data_date}) ')

In [ ]:
alldat['stationid'] = station

dt_end = alldat.index.max()
dt_start = alldat.index.min() - pd.Timedelta(minutes=interval)
timestart = dt_start.strftime('%Y%m%d%H%M')
timeend = dt_end.strftime('%Y%m%d%H%M')
file_name = f"{station}_{timestart:}_{timeend:}_raw.parquet"
file_path = clean_file_folder + "raw//" + file_name
alldat.to_parquet(file_path)
from datetime import datetime
today = datetime.now()
print(f'Data exported {today}')
print(f'Exported data to {file_path}')